# Tutorial 1: mock TCGA scVI embeddings and TME subtype prediction

This tutorial runs the complete development workflow using **synthetic expression and synthetic subtype labels only**. It visualizes cohort effects before and after scVI, maps a separate unseen cohort into the saved latent space, and predicts placeholder Bagaev-style TME subtypes.

> The requested term **PCCA** is interpreted here as **PCA** (principal component analysis). If a specific PCCA method was intended, that can be added as a separate analysis.

Install the package and tutorial dependencies from the repository root before running the notebook:

```bash
python -m pip install -e '.[all,tutorial]'
```

scVI training can use a GPU when available. The reduced epoch count below is suitable for a tutorial, not final model selection.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from tumor_subtyper import (
    generate_mock_data,
    get_embedding_scvi,
    load_classifier,
    load_expression_data,
    load_new_cohort,
    normalize_expression,
    predict_subtypes,
    train_pipeline,
)

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in {"tutorial", "tutorials"} else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "mock_tutorial"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "mock_tutorial"

## 1. Generate and load mock TCGA cohorts

Each training cohort is written to its own CSV. Cohort membership is recovered from the filename, while the synthetic Bagaev-style labels live in a separate CSV. The unseen cohort is stored in a subdirectory so the training loader cannot accidentally include it.

In [ ]:
mock_paths = generate_mock_data(
    DATA_DIR,
    n_cohorts=5,
    samples_per_cohort=60,
    n_genes=200,
    n_subtypes=4,
    random_state=RANDOM_STATE,
)
dataset = load_expression_data(DATA_DIR)

print(f"Expression matrix: {dataset.expression.shape[0]} samples x {dataset.expression.shape[1]} genes")
display(pd.crosstab(dataset.cohorts, dataset.labels))
display(dataset.expression.iloc[:5, :8])

### Raw-expression PCA and UMAP, colored by batch

We apply the same library-size plus `log1p` normalization used by the default pipeline, select the 100 most variable mock genes, and standardize them for visualization. These projections are descriptive only and are not classifier inputs.

In [ ]:
normalized_expression = normalize_expression(dataset.expression)
variable_genes = normalized_expression.var(axis=0).nlargest(100).index
raw_scaled = StandardScaler().fit_transform(normalized_expression.loc[:, variable_genes])

raw_pca = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(raw_scaled)
raw_umap = umap.UMAP(
    n_neighbors=20, min_dist=0.25, metric="euclidean", random_state=RANDOM_STATE
).fit_transform(raw_scaled)

raw_projection = pd.DataFrame(
    {
        "PCA_1": raw_pca[:, 0],
        "PCA_2": raw_pca[:, 1],
        "UMAP_1": raw_umap[:, 0],
        "UMAP_2": raw_umap[:, 1],
        "batch": dataset.cohorts.to_numpy(),
    },
    index=dataset.expression.index,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=raw_projection, x="PCA_1", y="PCA_2", hue="batch", s=35, ax=axes[0])
sns.scatterplot(data=raw_projection, x="UMAP_1", y="UMAP_2", hue="batch", s=35, ax=axes[1])
axes[0].set_title("Normalized mock expression: PCA by batch")
axes[1].set_title("Normalized mock expression: UMAP by batch")
for axis in axes:
    axis.legend(title="Cohort", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 2. Train scVI and inspect its transformed embeddings

The pipeline treats samples as AnnData observations and cohort as scVI's batch variable. It saves the reference model, manifest, cross-validation results, and final XGBoost classifier.

In [ ]:
training = train_pipeline(
    DATA_DIR,
    ARTIFACT_DIR,
    model_kind="xgboost",
    n_splits=5,
    n_latent=20,
    max_epochs=50,
    random_state=RANDOM_STATE,
    classifier_params={"n_estimators": 150},
)
train_embeddings = training.embeddings
display(training.classifier_result.fold_metrics)
display(train_embeddings.head())
print("scVI embedding shape:", train_embeddings.shape)

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(
    train_embeddings.iloc[:40],
    cmap="vlag",
    center=0,
    yticklabels=False,
    cbar_kws={"label": "Latent value"},
)
plt.title("scVI embeddings for the first 40 mock TCGA samples")
plt.xlabel("scVI latent dimension")
plt.ylabel("Mock TCGA samples")
plt.tight_layout()
plt.show()

## 3–4. UMAP of scVI embeddings by batch and synthetic TME subtype

A single UMAP fit is reused in both panels, so only the coloring changes. Effective correction should reduce separation driven purely by mock cohort while retaining shared subtype structure.

In [ ]:
latent_umap_model = umap.UMAP(
    n_neighbors=20, min_dist=0.25, metric="euclidean", random_state=RANDOM_STATE
)
latent_xy = latent_umap_model.fit_transform(train_embeddings)
latent_projection = pd.DataFrame(
    {
        "UMAP_1": latent_xy[:, 0],
        "UMAP_2": latent_xy[:, 1],
        "batch": dataset.cohorts.loc[train_embeddings.index].to_numpy(),
        "subtype": dataset.labels.loc[train_embeddings.index].to_numpy(),
    },
    index=train_embeddings.index,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=latent_projection, x="UMAP_1", y="UMAP_2", hue="batch", s=38, ax=axes[0])
sns.scatterplot(data=latent_projection, x="UMAP_1", y="UMAP_2", hue="subtype", s=38, ax=axes[1])
axes[0].set_title("scVI latent UMAP colored by batch")
axes[1].set_title("scVI latent UMAP colored by synthetic TME subtype")
for axis in axes:
    axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5. Load unseen data and extract embeddings with the trained scVI model

The new cohort is aligned to the saved training genes, normalized using the same recorded transform, and query-mapped into the reference latent space. The reference model is not retrained.

In [ ]:
new_expression = load_new_cohort(
    mock_paths.new_cohort_file, reference_genes=dataset.expression.columns
)
new_normalized = normalize_expression(new_expression)
test_embeddings = get_embedding_scvi(new_normalized, ARTIFACT_DIR / "scvi_model")

display(test_embeddings.head())
print("Unseen-cohort embedding shape:", test_embeddings.shape)

## 6. Predict TME subtypes from the extracted embeddings

In [ ]:
classifier = load_classifier(ARTIFACT_DIR / "classifier.joblib")
test_predictions = predict_subtypes(classifier, test_embeddings)
display(test_predictions.head(10))
display(test_predictions["predicted_subtype"].value_counts().rename("sample_count"))

## 7. Overlay unseen samples on the mock TCGA training embedding

For a fair visual comparison, UMAP is fit once on the concatenated training and test embeddings. This joint UMAP is only a visualization; subtype prediction above uses the original scVI coordinates directly.

In [ ]:
combined_embeddings = pd.concat([train_embeddings, test_embeddings], axis=0)
combined_group = pd.Series(
    ["Mock TCGA training"] * len(train_embeddings) + ["New cohort"] * len(test_embeddings),
    index=combined_embeddings.index,
    name="dataset",
)
combined_xy = umap.UMAP(
    n_neighbors=20, min_dist=0.25, metric="euclidean", random_state=RANDOM_STATE
).fit_transform(combined_embeddings)
combined_projection = pd.DataFrame(
    {"UMAP_1": combined_xy[:, 0], "UMAP_2": combined_xy[:, 1], "dataset": combined_group},
    index=combined_embeddings.index,
)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=combined_projection,
    x="UMAP_1",
    y="UMAP_2",
    hue="dataset",
    hue_order=["Mock TCGA training", "New cohort"],
    palette={"Mock TCGA training": "#B8B8B8", "New cohort": "#D62728"},
    size="dataset",
    sizes={"Mock TCGA training": 28, "New cohort": 55},
    alpha=0.85,
)
plt.title("New cohort overlaid on mock TCGA scVI embeddings")
plt.legend(title="Dataset", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 8. UMAP of unseen embeddings colored by predicted TME subtype

We retain the test-sample coordinates from the joint projection above, which makes this view directly comparable with the overlay.

In [ ]:
test_projection = combined_projection.loc[test_embeddings.index, ["UMAP_1", "UMAP_2"]].copy()
test_projection["predicted_subtype"] = test_predictions.loc[
    test_projection.index, "predicted_subtype"
]

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=test_projection,
    x="UMAP_1",
    y="UMAP_2",
    hue="predicted_subtype",
    palette="tab10",
    s=65,
    alpha=0.9,
)
plt.title("Unseen cohort colored by predicted synthetic TME subtype")
plt.legend(title="Predicted subtype", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Next steps

When approved real data become available, replace the cohort CSVs and label file while preserving the documented samples-by-genes schema. Before scientific use, validate the bulk normalization choice, scVI count-likelihood assumptions, feature identifiers, and generalization on genuinely held-out cohorts.